In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01. Data Exploration\n",
    "Exploring market data for Monte Carlo simulation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "from src.pipeline.data_fetcher import DataFetcher\n",
    "from src.pipeline.feature_engineering import FeatureEngineer\n",
    "\n",
    "%matplotlib inline\n",
    "sns.set_style('darkgrid')\n",
    "print('[INFO] Libraries loaded successfully')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fetcher = DataFetcher()\n",
    "tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']\n",
    "\n",
    "data = fetcher.get_historical_data(tuple(tickers), period='2y')\n",
    "if data is None or data.empty:\n",
    "    raise RuntimeError('No data returned. Provider errors: ' + str(fetcher.last_errors))\n",
    "\n",
    "missing = [t for t in tickers if t not in data.columns]\n",
    "if missing:\n",
    "    print('[WARNING] No data for:', missing)\n",
    "tickers = [t for t in tickers if t in data.columns]\n",
    "\n",
    "print('[INFO] Data shape:', data.shape)\n",
    "display(data.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 6))\n",
    "for ticker in tickers:\n",
    "    plt.plot(data.index, data[ticker], label=ticker, alpha=0.7)\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Price (local currency)')\n",
    "plt.title('Historical Prices')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "returns = data.pct_change().dropna()\n",
    "cumulative = (1 + returns).cumprod() - 1   # compounded, not a sum of simple returns\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "for ticker in tickers:\n",
    "    plt.plot(cumulative.index, cumulative[ticker], label=ticker, alpha=0.7)\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Cumulative Return')\n",
    "plt.title('Cumulative Returns')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "corr = returns.corr()\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')\n",
    "plt.title('Asset Correlation Matrix')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "vol = (returns.rolling(20).std() * np.sqrt(252)).dropna()\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "for ticker in tickers:\n",
    "    plt.plot(vol.index, vol[ticker], label=ticker, alpha=0.7)\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Annualized Volatility')\n",
    "plt.title('Rolling 20-Day Volatility')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "engineer = FeatureEngineer()\n",
    "features = engineer.create_features(data[tickers])\n",
    "normalized = engineer.normalize_features(features, fit=True)   # fit on training data only in real use\n",
    "\n",
    "print('[INFO] Features shape:', features.shape)\n",
    "print('[INFO] NaN count:', int(np.isnan(features.values).sum()))\n",
    "display(features.iloc[:, :8].tail())"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}